In [2]:
pip install flwr torch numpy

Note: you may need to restart the kernel to use updated packages.


In [4]:
pip install scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [6]:
import pandas as pd
client_df = pd.read_csv("C:/Users/saipr/OneDrive/Documents/Deep Learning/FL-IDS Project/client3_data.csv")

In [8]:
from sklearn.model_selection import train_test_split

def split_client_data(client_df, test_size=0.2, val_size=0.2):
    X = client_df.drop(columns=["Label"])
    y = client_df["Label"]

    X_temp, X_test, y_temp, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=42
    )

    val_ratio_adjusted = val_size / (1 - test_size)
    X_train, X_val, y_train, y_val = train_test_split(
        X_temp, y_temp, test_size=val_ratio_adjusted, stratify=y_temp, random_state=42
    )

    return X_train, X_val, X_test, y_train, y_val, y_test

X_train, X_val, X_test, y_train, y_val, y_test = split_client_data(client_df)

In [10]:
from model import LSTMIDS

In [12]:
import torch
from torch.utils.data import TensorDataset, DataLoader

def get_tensor_loaders(X_train, y_train, X_val, y_val, X_test, y_test, batch_size=32):
    train_dataset = TensorDataset(torch.tensor(X_train.values, dtype=torch.float32),
                                   torch.tensor(y_train.values, dtype=torch.float32))
    
    val_dataset = TensorDataset(torch.tensor(X_val.values, dtype=torch.float32),
                                 torch.tensor(y_val.values, dtype=torch.float32))

    test_dataset = TensorDataset(torch.tensor(X_test.values, dtype=torch.float32),
                                  torch.tensor(y_test.values, dtype=torch.float32))

    return DataLoader(train_dataset, batch_size=batch_size, shuffle=True), \
           DataLoader(val_dataset, batch_size=batch_size), \
           DataLoader(test_dataset, batch_size=batch_size)

train_loader, val_loader, test_loader = get_tensor_loaders(
    X_train, y_train, X_val, y_val, X_test, y_test
)

In [14]:
import flwr as fl
import torch
from torch import nn
from sklearn.metrics import precision_score, recall_score, f1_score
import csv
import os

# Do NOT initialize CrypTen in the client — server handles SMPC!

class IDSClient(fl.client.NumPyClient):
    def __init__(self, model, train_loader, val_loader):
        self.model = model
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.loss_fn = nn.BCELoss()
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=0.001)

    def get_parameters(self, config=None):
        return [val.cpu().detach().numpy() for val in self.model.parameters()]

    def set_parameters(self, parameters):
        for param, new_val in zip(self.model.parameters(), parameters):
            param.data = torch.tensor(new_val, dtype=param.data.dtype)

    def fit(self, parameters, config):
        self.set_parameters(parameters)
        self.model.train()
        for x_batch, y_batch in self.train_loader: 
            if len(x_batch.shape) == 2:
                x_batch = x_batch.unsqueeze(1)
            self.optimizer.zero_grad()
            y_pred = self.model(x_batch).squeeze()
            loss = self.loss_fn(y_pred, y_batch)
            loss.backward()
            self.optimizer.step()

        # Return plaintext parameters (SMPC happens on the server)
        updated_params = self.get_parameters()
        return updated_params, len(self.train_loader.dataset), {}

    def log_metrics(self, accuracy, precision, recall, f1, round_num):
        filename = f"client1_metrics_rounds.csv"
        file_exists = os.path.isfile(filename)

        with open(filename, mode='a', newline='') as file:
            writer = csv.writer(file)
            if not file_exists:
                writer.writerow(["Round", "Accuracy", "Precision", "Recall", "F1"])
            writer.writerow([round_num, accuracy, precision, recall, f1])

    def evaluate(self, parameters, config):
        self.set_parameters(parameters)
        self.model.eval()
        loss, correct, total = 0.0, 0, 0
        all_preds, all_labels = [], []

        with torch.no_grad():
            for x_batch, y_batch in self.val_loader:
                if len(x_batch.shape) == 2:
                    x_batch = x_batch.unsqueeze(1)
                y_pred = self.model(x_batch).squeeze()
                loss += self.loss_fn(y_pred, y_batch).item()

                preds = (y_pred >= 0.5).float()
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(y_batch.cpu().numpy())

                correct += (preds == y_batch).sum().item()
                total += y_batch.size(0)

        accuracy = correct / total
        precision = precision_score(all_labels, all_preds, zero_division=0)
        recall = recall_score(all_labels, all_preds, zero_division=0)
        f1 = f1_score(all_labels, all_preds, zero_division=0)

        print(f"Client1 Metrics: Accuracy={accuracy:.4f}, Precision={precision:.4f}, Recall={recall:.4f}, F1={f1:.4f}")

        round_num = int(config.get("round", -1))
        self.log_metrics(accuracy, precision, recall, f1, round_num)

        return float(loss), total, {
            "accuracy": float(accuracy),
            "precision": float(precision),
            "recall": float(recall),
            "f1_score": float(f1)
        }

# Create model and start client
from model import LSTMIDS  # Your shared model definition

model = LSTMIDS(input_size=25)
client = IDSClient(model, train_loader, val_loader)

fl.client.start_client(
    server_address="192.168.1.80:8080",
    client=client.to_client()
)

C:\Users\saipr\anaconda3\envs\crypten_env\lib\site-packages\torch\nn\modules\rnn.py:62: UserWarning: dropout option adds dropout after all but last recurrent layer, so non-zero dropout expects num_layers greater than 1, but got dropout=0.2 and num_layers=1
  warnings.warn("dropout option adds dropout after all but last "
INFO :      
INFO :      Received: train message a0767e76-bfaa-4547-94b0-f5fb43ec2481
INFO :      Sent reply
INFO :      
INFO :      Received: evaluate message 93796282-c40f-4207-89c0-8140e617bd60
INFO :      Sent reply
INFO :      
INFO :      Received: train message a8898ea2-bfa8-4fb4-a7d4-b16cb37cc2e2


Client1 Metrics: Accuracy=0.9749, Precision=0.9501, Recall=0.9792, F1=0.9644


INFO :      Sent reply
INFO :      
INFO :      Received: evaluate message 4300ad04-252d-4573-ab48-c0de5b8f8c75
INFO :      Sent reply
INFO :      
INFO :      Received: train message 507451fb-f724-47ae-855e-f3270c42f35f


Client1 Metrics: Accuracy=0.9795, Precision=0.9610, Recall=0.9809, F1=0.9708


INFO :      Sent reply
INFO :      
INFO :      Received: evaluate message 98ef204a-4230-49fd-ab64-5e9247bfb728
INFO :      Sent reply
INFO :      
INFO :      Received: train message 5562176a-25ef-4d59-b2a3-82d58923dd49


...

INFO :      Sent reply
INFO :      
INFO :      Received: evaluate message 11e683cb-fcfe-4b54-ae50-ad2ff42d14e9
INFO :      Sent reply
INFO :      
INFO :      Received: train message e9cfec06-a848-4423-b9c7-764c3a6598a7


Client1 Metrics: Accuracy=0.9920, Precision=0.9894, Recall=0.9875, F1=0.9884


INFO :      Sent reply
INFO :      
INFO :      Received: evaluate message 363e546e-ec9e-4819-8499-4aadbbf3e0ed
INFO :      Sent reply
INFO :      
INFO :      Received: reconnect message c8f41e18-a489-433f-a85c-1c1d7fc13ba3


Client1 Metrics: Accuracy=0.9917, Precision=0.9873, Recall=0.9887, F1=0.9880


INFO :      Disconnect and shut down
